# Practical PyTorch · Part 18 — Companion Notebook

**Triage the Inbox: Build a Feedback Sorter with DistilBERT.**

The text-classification capstone. Batch a pile of feedback through `pipeline("sentiment-analysis")` (DistilBERT), route low-confidence cases to a human, and wrap it all in a Gradio app.

In [ ]:
!pip install -q transformers gradio

## 1. One call, a whole batch

Hand the pipeline a *list* and it classifies every item in one pass — list in, list of `{label, score}` dicts out.

In [ ]:
from transformers import pipeline

clf = pipeline("sentiment-analysis")   # the DistilBERT SST-2 model from Part 15

feedback = [
    "Absolutely loved it — would buy again.",
    "The app crashed twice and ate my order.",
    "It arrived. It works. That's about it.",
]
for text, r in zip(feedback, clf(feedback)):
    print(f"{r['label']:>8}  {r['score']:.2f}  {text}")

## 2. A verdict, plus a way to say "not sure"

A binary model shoves every input into POSITIVE or NEGATIVE, often confidently. A **threshold** routes anything below the line to a human instead of mislabeling it.

In [ ]:
def triage(text, threshold):
    items = [line.strip() for line in text.splitlines() if line.strip()]
    rows, counts = [], {"POSITIVE": 0, "NEGATIVE": 0, "needs a human": 0}
    for item, r in zip(items, clf(items)):
        verdict = r["label"] if r["score"] >= threshold else "needs a human"
        counts[verdict] += 1
        rows.append([item, verdict, round(r["score"], 3)])
    return rows, counts

## 3. Wrap it in a UI

Same `gr.Interface` move as Part 12 — two inputs and two outputs, so each is a list.

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=triage,
    inputs=[
        gr.Textbox(lines=10, label="Paste feedback — one item per line"),
        gr.Slider(0.5, 1.0, value=0.95, step=0.01, label="Confidence threshold"),
    ],
    outputs=[
        gr.Dataframe(headers=["feedback", "verdict", "confidence"], label="Triaged"),
        gr.Label(label="Distribution"),
    ],
    title="Feedback Triage",
    description="Sort a pile of feedback into positive, negative, and needs-a-human.",
)
demo.launch()

**Know its limits:** binary (no real neutral), overconfident (trust the ordering, not the exact score), trained on movie reviews (weaker off-domain), and sentiment is the wrong question for "what is this about?" The threshold and the *needs a human* bucket are where the tool stays honest.

**Next: Part 19 — Beyond pipeline(): Taking the Wheel with from_pretrained.**